# Finite-ground CPW — Q3D AC/RL

Extract AC R/L and capacitance from the finite-ground CPW. See the [Q3D
AC/RL requirements](../../../simulation/requirements/q3d-acrl.html). The
public coupon is 500 µm long, with a 10 µm signal, 6 µm gaps and 80 µm
grounds. This current-PDK case selects Al 100 nm on Si 500 µm,
explicitly replacing the prior Nb 200 nm model. Source-free body-v3
records mirror the coupon rectangles; `geometry_source=None` is
deliberate, not a helper-produced source binding.

Use [SCGSim
2.0.0rc1](https://github.com/OrPenStrike/scgsim/tree/85d8bedb733b1e17737d935029618351b30ea5de)
in a separately selected consumer environment. The exact source revision
is recorded in the controls below. Execution is disabled; no native
solve or returned-result claim is made. `AUTHORING_ONLY` constructs
non-solving inputs; preparation/run are explicit, and analysis reads an
independent returned directory.

## Setup and Imports

In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path
from uuid import uuid4

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    AedtResources,
    MatrixRunControl,
    PdkMaterial,
    PhysicalLayerSpec,
    Q3dBodySpec,
    Q3dNetSpec,
    Q3dSpec,
    prepare_handoff,
    resolve_results,
)

import orpen_sc_pdk
from orpen_sc_pdk.cells import finite_ground_cpw_coupon
from orpen_sc_pdk.materials import get_material_records
from orpen_sc_pdk.tech import get_single_die_layer_stack

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "AUTHORING_ONLY"  # AUTHORING_ONLY | prepare_handoff | run | analyze_handoff
RUN_ID = f"cpw_q3d_{uuid4().hex}"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "CpwFiniteGround"
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = None  # Set independently to an immutable returned-run directory.
SCGSIM_RUNTIME_REVISION = "85d8bedb733b1e17737d935029618351b30ea5de"
RESOURCES = AedtResources(cores=4, ram_limit_percent=30)
MODELING = "solid"
SOLVE_AC_RL = True
# CPW trace length along X (um).
trace_length_um = 500.0
# Signal conductor width (um).
signal_width_um = 10.0
# Gap from signal to each ground conductor (um).
gap_um = 6.0
# Width of each ground conductor (um).
ground_width_um = 80.0
# Metal thickness (um).
metal_thickness_um = 0.1
# Substrate thickness below the metal (um).
substrate_thickness_um = 500.0
# AEDT version used for project generation.
aedt_version = "2024.2"
# AEDT project name.
project_name = RUN_ID
# AEDT design name.
design_name = "CpwFiniteGroundQ3d"
# Padding around the model in -X, +X, -Y, +Y, -Z, +Z directions (um).
region_padding_um = (0.0, 0.0, 100.0, 100.0, 1000.0, 1000.0)
# Matrix solve frequency (GHz).
frequency_ghz = 6.0
# Maximum adaptive passes.
maximum_passes = 3

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    orpen_sc_pdk.activate()
    gf.clear_cache()
    coupon = finite_ground_cpw_coupon()
    layer_stack = get_single_die_layer_stack()
    material_records = get_material_records()

## Initialize AEDT Project / App

In [ ]:
# Only the generated native runner owns an AEDT session.

## Import GDS and Build the HFSS/Q3D/Q2D Model

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    # Source-free body-v3 records mirror the four GDSFactory rectangles exactly.
    # Body IDs are authored identities, not guesses about native imported names.
    x = trace_length_um / 2
    s = signal_width_um / 2
    g = s + gap_um
    w = ground_width_um
    bodies = (
        Q3dBodySpec(
            "SignalBody",
            ((-x, -s), (x, -s), (x, s), (-x, s)),
            (),
            0.0,
            metal_thickness_um,
            "Al",
            "signal",
            "Signal",
            physical_layer_id="D0_TOP_M1",
        ),
        Q3dBodySpec(
            "GroundLeftBody",
            ((-x, g), (x, g), (x, g + w), (-x, g + w)),
            (),
            0.0,
            metal_thickness_um,
            "Al",
            "ground",
            "GroundLeft",
            physical_layer_id="D0_TOP_M1",
        ),
        Q3dBodySpec(
            "GroundRightBody",
            ((-x, -g - w), (x, -g - w), (x, -g), (-x, -g)),
            (),
            0.0,
            metal_thickness_um,
            "Al",
            "ground",
            "GroundRight",
            physical_layer_id="D0_TOP_M1",
        ),
        Q3dBodySpec(
            "SubstrateBody",
            ((-x, -g - w), (x, -g - w), (x, g + w), (-x, g + w)),
            (),
            -substrate_thickness_um,
            0.0,
            "Si",
            "substrate",
            None,
            physical_layer_id="D0_SUBSTRATE",
        ),
    )

## Geometry Verification

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(coupon)
    display({"bbox_um": str(coupon.dbbox())})

## Materials and Boundaries

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    material_records = get_material_records()
    materials = {
        material_id: PdkMaterial(
            material_id,
            material_records[material_id]["material_kind"],
            material_records[material_id]["is_superconducting"],
            material_records[material_id]["aedt_library_name"],
        )
        for material_id in ("vacuum", "Si", "Al")
    }

## Ports / Nets / Excitations

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    # Net names, roles, object members, and terminal directions define Q3D excitations.
    nets = (
        # Net name, net role, member objects, and optional terminal object/directions.
        Q3dNetSpec("Signal", "Signal", ("SignalBody",), "SignalBody", "-X", "SignalBody", "+X"),
        Q3dNetSpec("GroundLeft", "Ground", ("GroundLeftBody",)),
        Q3dNetSpec("GroundRight", "Ground", ("GroundRightBody",)),
    )

## Simulation Setup

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    spec = Q3dSpec(
        modeling=MODELING,
        project_name=project_name,
        physical_layers=(
            PhysicalLayerSpec("D0_TOP_M1", "stack_film", 0.0, metal_thickness_um, "lower"),
            PhysicalLayerSpec("D0_SUBSTRATE", "retained_solid", -substrate_thickness_um, 0.0),
        ),
        design_name=design_name,
        materials=materials,
        vacuum_material_id="vacuum",
        bodies=bodies,
        nets=nets,
        geometry_source=None,
        solve_ac_rl=SOLVE_AC_RL,
        run_control=MatrixRunControl("Setup1", frequency_ghz, maximum_passes),
        region_padding_um=region_padding_um,
        aedt_version=aedt_version,
    )

## Simulation Configuration

In [ ]:
if WORKFLOW_ACTION != "analyze_handoff":
    display(spec)

## Solve and Export

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = prepare_handoff(spec=spec, output_dir=RUN_DIR, resources=RESOURCES)
    display(HANDOFF)
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    result = resolve_results(Path(RETURNED_RUN_DIR))
    result.show_all_results(show_details=True)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display(result.physics_results())

### Simulation Performance / Benchmarks

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display(result.simulation_benchmark())

## Save and Release AEDT

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display({"project": result.project_path, "receipt": result.receipt_path})
elif WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(HANDOFF.archive_path)